In [ ]:
import pandas as pd
import matplotlib.pyplot as plt 
import numpy as np


In [ ]:
water_content = pd.read_csv('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/good_water_contents/tot_water_content.csv', index_col=0)
water_content.index = water_content.index * (5000/np.max(water_content.index)) 
water_content.insert(0, 'time', water_content.index)




In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Example DataFrame with 10 columns
# Replace this with your actual DataFrame
# data = {
#     'time': np.linspace(0, 5000, 50),  # Time column
# }
# for i in range(1, 11):
#     water_content[f'Column_{i}'] = np.random.normal(1.0, 0.1, 50)  # Example columns

# df = pd.DataFrame(data)

# Bootstrapping Function
def bootstrap(data, n_iterations=10000, confidence_level=0.95):
    boot_means = np.array([
        np.mean(np.random.choice(data, size=len(data), replace=True))
        for _ in range(n_iterations)
    ])
    lower_bound = np.percentile(boot_means, (1 - confidence_level) / 2 * 100)
    upper_bound = np.percentile(boot_means, (1 + confidence_level) / 2 * 100)
    return np.mean(boot_means), lower_bound, upper_bound

# Window Size for Analysis
window_size = 10  # Adjust as needed

# Store Results for Each Column
results = {}

for column in water_content.columns[1:]:  # Skip the 'time' column
    column_results = []
    for i in range(0, len(water_content), window_size):
        window_data = water_content[column].iloc[i:i + window_size]
        if len(window_data) > 0:  # Ensure the window has data
            mean, lower, upper = bootstrap(window_data)
            column_results.append({
                'start_time': water_content['time'].iloc[i],
                'end_time': water_content['time'].iloc[min(i + window_size - 1, len(water_content) - 1)],
                'mean_time': ((water_content['time'].iloc[i])+(water_content['time'].iloc[min(i + window_size - 1, len(water_content) - 1)]))/2,
                'mean': mean,
                'lower': lower,
                'upper': upper
            })
    results[column] = pd.DataFrame(column_results)

# Plot Results for All Columns
plt.figure(figsize=(7, 5))

colors = [
    # Set 2: Greens (lightest to darkest)
    "#A1D99B", "#74C476", "#31A354", "#2CA02C", "#006D2C",
    # Set 3: Purples (lightest to darkest)
    '#E5D8F7', '#C4A4E8', '#A87EDA', '#7E4AB8', '#4B237A',
    # Set 4: Oranges (lightest to darkest)
    "#FDD0A2", "#FDAE6B", "#FF7F0E", "#E6550D"]
loop_num=0
for column, result_df in results.items():
    color = colors[loop_num]
    if len(column) < 3:
        plt.plot(result_df['mean_time'], result_df['mean'], label=f'{column}', marker='o', color=color)
        plt.fill_between(result_df['mean_time'], result_df['lower'], result_df['upper'], alpha=0.3, color=color)#, label=f'{column} 95% CI')
        loop_num += 1
    else:
        if column == 'GGGXGXGGG':
            label = 'X2'
        elif column == 'GXGGGGGXG':
            label = 'X2S'
        elif column == 'GGGXXXGGG':
            label = 'X3'
        else:
            label = 'X3S'
        plt.plot(result_df['mean_time'], result_df['mean'], label=f'{label}', marker='o', color=color)
        plt.fill_between(result_df['mean_time'], result_df['lower'], result_df['upper'], alpha=0.3, color=color)#, label=f'{column} 95% CI')
        loop_num += 1

plt.xlabel('Time (ns)', fontsize=14)
plt.ylabel('Water:Peptide Mass Ratio', fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.ylim(0,3)
plt.xlim(0,5000)
# plt.title('Bootstrapping Convergence Analysis for All Columns')
plt.legend(ncols=3)

plt.savefig('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/convergence/pngs/mass_ratio_convergence.png', dpi=500, transparent=True)
plt.savefig('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/convergence/pdfs/mass_ratio_convergence.pdf', dpi=500, transparent=True)

plt.show()

In [ ]:
colors = [
    # Set 1: Blues (lightest to darkest)
    "#A6CEE3", "#6BAED6", "#3182BD", "#1F77B4", "#08519C",
    # Set 2: Greens (lightest to darkest)
    "#A1D99B", "#74C476", "#31A354", "#2CA02C", "#006D2C",
    # Set 3: Oranges (lightest to darkest)
    "#FDD0A2", "#FDAE6B", "#FF7F0E", "#E6550D"]

water_content.plot(color=colors, figsize=[7,5])
plt.legend(loc='lower right',ncols=3, fontsize=8)
plt.xlabel('Time (ns)', fontsize=14)
plt.ylabel('Water:Peptide Mass Ratio', fontsize=14)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14);

plt.savefig('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/water/pdfs/water_pep_ratio_traj.pdf', dpi=500, transparent=True)
plt.savefig('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/water/pngs/water_pep_ratio_traj.png', dpi=500, transparent=True)


In [ ]:
water_content[water_content.columns[1:]]

In [ ]:
colors = [
    # Set 2: Greens (lightest to darkest)
    "#2CA02C", "#2CA02C", "#2CA02C", "#2CA02C", "#2CA02C",
    # Set 3: Purples (lightest to darkest)
    '#7E4AB8', '#7E4AB8', '#7E4AB8', '#7E4AB8', '#7E4AB8',
    # Set 4: Oranges (lightest to darkest)
    "#FF7F0E", "#FF7F0E", "#FF7F0E", "#FF7F0E"]

fig = plt.subplots(figsize=[7,6])

cleaned_data = [water_content[col].dropna() for col in water_content.columns[1:]]

violins = plt.violinplot(cleaned_data, showmeans=False, showmedians=False)

# Customize each violin with colors
for i, body in enumerate(violins['bodies']):
    body.set_facecolor(colors[i])  # Set the fill color
    body.set_edgecolor('none')   # Optional: Set edge color
    body.set_alpha(0.7)           # Optional: Adjust transparency
    
violins['cbars'].set_colors(colors)

violins['cmins'].set_colors(colors)
    
violins['cmaxes'].set_colors(colors)

plt.xticks(ticks=range(1, len(water_content.columns[1:]) + 1), labels=water_content.columns[1:], fontsize=12, rotation=45);
plt.yticks(fontsize=12)
plt.ylabel('Water:Peptide Mass Ratio', fontsize=14)
plt.xlabel('System', fontsize=14);

plt.tight_layout()

plt.savefig('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/water/full_water_content_violin.pdf', dpi=500, transparent=True)
plt.savefig('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/water/full_water_content_violin.png', dpi=500, transparent=True)

plt.show()